# Lesson 3 Assignment

In this lab assignment, you will implement a simplified version of Random Forest classifier and practice how to use and fine-tune Random Forest, Extra Trees, and Gradient Boosted Trees. You will then compare the model performance of various classifiers on internet ad dataset.

In [1]:
# import packages
%matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.utils import resample

# make this notebook's output stable across runs
np.random.seed(0)

## Data Set Information:

This dataset represents a set of possible advertisements on Internet pages. The features encode the geometry of the image (if available) as well as phrases occuring in the URL, the image's URL and alt text, the anchor text, and words occuring near the anchor text. The task is to predict whether an image is an advertisement ("ad") or not ("nonad"). Additional information can be found [here](https://archive.ics.uci.edu/ml/datasets/internet%2Badvertisements).

## Attribute Information:

The dataset has 3 continous (height, width, aratio) and 1555 binary (urls, tags, captions) features. 

## Source:

Creator & donor: Nicholas Kushmerick <nick '@' ucd.ie>

In [2]:
# Load the data
internetAd = pd.read_csv('Internet_Ad_Data.csv', sep=',', error_bad_lines=False)
print(internetAd.info())
internetAd.head(20)

/tmp/ipykernel_24995/3747926279.py:2: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  internetAd = pd.read_csv('Internet_Ad_Data.csv', sep=',', error_bad_lines=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3279 entries, 0 to 3278
Columns: 1559 entries, height to Target
dtypes: int64(1554), object(5)
memory usage: 39.0+ MB
None


/tmp/ipykernel_24995/3747926279.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  internetAd = pd.read_csv('Internet_Ad_Data.csv', sep=',', error_bad_lines=False)


,height,width,aratio,local,url*images+buttons,url*likesbooks.com,url*www.slake.com,url*hydrogeologist,url*oso,url*media,...,caption*home,caption*my,caption*your,caption*in,caption*bytes,caption*here,caption*click,caption*for,caption*you,Target
0,125,125,1.0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
1,57,468,8.2105,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
2,33,230,6.9696,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
3,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
4,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
5,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
6,59,460,7.7966,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
7,60,234,3.9,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
8,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
9,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.


## Question 1: Prepare and impute missing values with the median (missing values for this dataset are \?, nonad. ad.)

In [3]:
for column in internetAd.columns:
    if internetAd[column].dtype == object:
        print(column)
    else:
        pass

height
width
aratio
local
Target


In [4]:
obj_feats = ["height","width","aratio","local","Target"] #list of features that have dtype object

In [5]:
for feature in internetAd.columns: internetAd[feature] = internetAd[feature].replace(regex="\?$",value=np.NAN)        #Removes _? and replaces it with NAN
internetAd["Target"] = internetAd["Target"].replace(to_replace={"nonad.","ad."}, value={0,1})                         #Encodes the Target column
for feature in obj_feats: internetAd[feature] = internetAd[feature].astype(float)                                     #Changes dtype of columns that contain objects to columns that contain floats
for feature in internetAd.columns: internetAd[feature] = internetAd[feature].fillna(value=internetAd.mean()[feature]) #Fills in any missing values with the average of that feature

## Question 2: Split dataset into training and test set

In [6]:
from sklearn.model_selection import train_test_split

X = internetAd.drop(columns="Target")
y = internetAd["Target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## Question 3: Train and evaluate a randomeforrest classifier using the following gridsearch parameters:
- "max_depth": [2, 4],
- "min_samples_split": [0.05, 0.1, 0.2]

In [7]:
parameters = {"max_depth":[2,4], "min_samples_split":[0.5,0.1,0.2]}
clf = RandomForestClassifier()
dtc_grid = GridSearchCV(clf, parameters) #unsure if I should use clf.estimator_ for estimator or just clf
rfor = dtc_grid.fit(X_train,y_train)

In [8]:

# make predictions with the trained random forest
test_z = rfor.predict(X_test)
test_z_prob = rfor.predict_proba(X_test)

# evaluate the model performance - ACCURACY AND ROC AUC
y_true = y_test
acc = accuracy_score(y_true, test_z)
auc = roc_auc_score(y_true, test_z)
print("Accuracy: {}%. \n ROC AUC: {}.".format(round(acc*100,2), round(auc,2)))

Accuracy: 90.49%. 
 ROC AUC: 0.7.


## Question 4: Train and evaluate a ExtraTrees classifier using the following gridsearch parameters:
- "max_depth": [2, 4],
- "min_samples_split": [0.05, 0.1, 0.2]

In [9]:
parameters = {"max_depth":[2,4], "min_samples_split":[0.5,0.1,0.2]}
clf = ExtraTreesClassifier()
dtc_grid = GridSearchCV(clf, parameters) #unsure if I should use clf.estimator_ for estimator or just clf
efor = dtc_grid.fit(X_train,y_train)

In [10]:

# make predictions with the trained random forest
test_z = efor.predict(X_test)
test_z_prob = efor.predict_proba(X_test)

# evaluate the model performance - ACCURACY AND ROC AUC
y_true = y_test
acc = accuracy_score(y_true, test_z)
auc = roc_auc_score(y_true, test_z)
print("Accuracy: {}%. \n ROC AUC: {}.".format(round(acc*100,2), round(auc,2)))

Accuracy: 88.73%. 
 ROC AUC: 0.64.


## Question 5: Train and evaluate a Gradient Boosted Trees classifier using the following gridsearch parameters:
- "max_depth": [2, 4],
- "min_samples_split": [0.05, 0.1, 0.2]

In [11]:
parameters = {"max_depth":[2,4], "min_samples_split":[0.5,0.1,0.2]}
clf = GradientBoostingClassifier()
dtc_grid = GridSearchCV(clf, parameters) #unsure if I should use clf.estimator_ for estimator or just clf
gbtr = dtc_grid.fit(X_train,y_train)

In [12]:

# make predictions with the trained random forest
test_z = gbtr.predict(X_test)
test_z_prob = gbtr.predict_proba(X_test)

# evaluate the model performance - ACCURACY AND ROC AUC
y_true = y_test
acc = accuracy_score(y_true, test_z)
auc = roc_auc_score(y_true, test_z)
print("Accuracy: {}%. \n ROC AUC: {}.".format(round(acc*100,2), round(auc,2)))

Accuracy: 96.12%. 
 ROC AUC: 0.9.


## [Bonus] Question 6: Which algorithm performed better and why?


The Gradient Boosted Trees classifier performed FAR better than the ExtraTrees classifier and the regular RandomForestClassifier. Using both Accuracy and ROC AUC as metrics for performance it's clearly superior in this case with this dataset. I imagine it has to do with the fact Gradient Boosting doesn't work on subsamples of the dataset whereas both ExtraTrees and RandomForests do, the latter two also use averaging to determine the improvements in predictive accuracy whereas Gradient Boosted Trees utilizes the gradient of the loss function in a gradient descent fashion which seems more appropriate with such a high dimensional dataset.

## Question 7. Create a new text cell in your Notebook: Complete a 50-100 word summary (or short description of your thinking in applying this week's learning to the solution) of your experience in this assignment. Include:
                                                                      
* What was your incoming experience with this model, if any? 
* What steps you took, what obstacles you encountered.
* How you link this exercise to real-world, machine learning problem-solving. (What steps were missing? What else do you need to learn?) 
> This summary allows your instructor to know how you are doing and allot points for your effort in thinking and planning, and making connections to real-world work.

My only experience with classifiers that use trees is from last week, so in general I had no incoming experience with the model. I utilized the sklearn documentation to overcome the obstacles I came across, for instance making sure I understood what the appropriate parameters were for GridSearchCV. I wonder if this sort of approach could be used to sort Astronomical catalogues (of stars, and galaxies) which often have several entries for the same object but with different names and slightly different properties (observations made by different telescopes, published in different catalogues, giving them different names and slightly different values for things like RA and DEC (coordinates in sky)), I think an algorithm could sort whether or not two catalogues have duplicate entries in the same way the above algorithm is able to identify ads. I may be too optimistic in envisaging such an application, I ran into issues when I was writing a paper one time and this was the source of signifigant confusion. 